In [ ]:
# STEP 1 - Setup

import os
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# STEP 2 - Get GasHisSDB file list from Figshare

import requests
import pandas as pd

ARTICLE_ID = 15066147
BASE_URL = "https://api.figshare.com/v2"

metadata_url = f"{BASE_URL}/articles/{ARTICLE_ID}/files"
metadata = requests.get(metadata_url).json()

files_df = pd.DataFrame(metadata)

print("Number of files found:", len(files_df))
print(files_df[["id", "name", "size", "download_url"]])

In [ ]:
# STEP 2.1 - Download GasHisSDB.rar

import requests
from pathlib import Path
from tqdm import tqdm

RAR_PATH = Path("/content/GasHisSDB.rar")

download_url = files_df.loc[0, "download_url"]
file_size = int(files_df.loc[0, "size"])

print("Download URL:", download_url)
print("File size:", round(file_size / (1024**3), 2), "GB")

if RAR_PATH.exists():
    print("File already exists:", RAR_PATH)
    print("Current size:", round(RAR_PATH.stat().st_size / (1024**3), 2), "GB")
else:
    response = requests.get(download_url, stream=True)
    response.raise_for_status()

    with open(RAR_PATH, "wb") as f:
        progress = tqdm(
            total=file_size,
            unit="B",
            unit_scale=True,
            desc="Downloading GasHisSDB.rar"
        )

        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                progress.update(len(chunk))

        progress.close()

print("Download finished.")
print("Saved path:", RAR_PATH)
print("Final size:", round(RAR_PATH.stat().st_size / (1024**3), 2), "GB")

In [ ]:
# STEP 2.2 - Install unrar

!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# STEP 2.3 - List archive contents

RAR_PATH = "/content/GasHisSDB.rar"

!unrar lb "{RAR_PATH}" | head -100

In [ ]:
# STEP 2.4 - Check available subset folders inside archive

RAR_PATH = "/content/GasHisSDB.rar"

!unrar lb "{RAR_PATH}" | cut -d'/' -f2 | sort | uniq -c

In [ ]:
# STEP 2.5 - Search for 160x160 subset

!unrar lb "{RAR_PATH}" | grep "^GasHisSDB/160/" | head -50

In [ ]:
# STEP 2.6 - Extract only the 160x160 subset

RAR_PATH = "/content/GasHisSDB.rar"
EXTRACT_DIR = "/content/GasHisSDB_extracted"

!mkdir -p "{EXTRACT_DIR}"
!unrar x -y "{RAR_PATH}" "GasHisSDB/160/*" "{EXTRACT_DIR}/"

In [ ]:
# STEP 2.7 - Check extracted 160x160 subset structure

DATA_ROOT = "/content/GasHisSDB_extracted/GasHisSDB/160"

import os

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
print("DATA_ROOT:", DATA_ROOT)

for root, dirs, files in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level

    if level <= 2:
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = ' ' * 2 * (level + 1)
        for f in files[:5]:
            print(f"{sub_indent}{f}")

In [ ]:
# STEP 2.8 - Count images and class distribution

from pathlib import Path
import pandas as pd

IMG_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

all_images = []
for ext in IMG_EXTENSIONS:
    all_images.extend(list(Path(DATA_ROOT).rglob(f"*{ext}")))

all_images = [str(p) for p in all_images]

def infer_label(path):
    lower_path = path.lower()


    if "abnormal" in lower_path:
        return 1, "abnormal"
    elif "normal" in lower_path:
        return 0, "normal"
    else:
        return None, "unknown"

records = []

for p in all_images:
    label, class_name = infer_label(p)
    records.append({
        "path": p,
        "label": label,
        "class_name": class_name
    })

df = pd.DataFrame(records)

print("Total images found:", len(df))
print("\nClass distribution:")
print(df["class_name"].value_counts())

print("\nUnknown labels:", (df["class_name"] == "unknown").sum())

df.head()

In [ ]:
# STEP 3 - Show sample normal and abnormal images

import matplotlib.pyplot as plt
import tensorflow as tf

known_df = df[df["class_name"] != "unknown"].copy()

sample_df = (
    known_df
    .groupby("class_name", group_keys=False)
    .apply(lambda x: x.sample(min(5, len(x)), random_state=SEED))
)

plt.figure(figsize=(14, 6))

for i, row in enumerate(sample_df.itertuples(), 1):
    img = tf.keras.utils.load_img(row.path)

    plt.subplot(2, 5, i)
    plt.imshow(img)
    plt.title(row.class_name)
    plt.axis("off")

plt.suptitle("Sample Images from GasHisSDB 160x160 Subset", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 4 - Stratified train / validation / test split

from sklearn.model_selection import train_test_split

df_clean = df[df["class_name"] != "unknown"].copy()
df_clean = df_clean.sample(frac=1, random_state=SEED).reset_index(drop=True)

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    stratify=df_clean["label"],
    random_state=SEED
)

# Second split: temporary 30% into 15% validation and 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Total images:", len(df_clean))
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nTraining class distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation class distribution:")
print(val_df["class_name"].value_counts())

print("\nTest class distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
# STEP 5 - Compute class weights for imbalanced dataset

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("Class weights:")
print("Normal class weight:", class_weights[0])
print("Abnormal class weight:", class_weights[1])

In [ ]:
# STEP 6 - Create TensorFlow data pipeline

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.10, 0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="data_augmentation")

def add_augmentation(image, label):
    image = data_augmentation(image, training=True)
    return image, label

def create_dataset(dataframe, training=False):
    paths = dataframe["path"].values
    labels = dataframe["label"].values.astype("float32")

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.shuffle(buffer_size=1000, seed=SEED)
        dataset = dataset.map(add_augmentation, num_parallel_calls=AUTOTUNE)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df, training=False)
test_ds = create_dataset(test_df, training=False)

print("Datasets created successfully.")
print("Train batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Test batches:", len(test_ds))

In [ ]:
# STEP 7 - Visualize augmented training images

for images, labels in train_ds.take(1):
    plt.figure(figsize=(12, 8))

    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        plt.title("Abnormal" if labels[i].numpy() == 1 else "Normal")
        plt.axis("off")

    plt.suptitle("Augmented Training Images", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 7.1 - Recreate tf.data pipeline with value clipping after augmentation

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.10, 0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="data_augmentation")

def add_augmentation(image, label):
    image = data_augmentation(image, training=True)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def create_dataset(dataframe, training=False):
    paths = dataframe["path"].values
    labels = dataframe["label"].values.astype("float32")

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.shuffle(buffer_size=1000, seed=SEED)
        dataset = dataset.map(add_augmentation, num_parallel_calls=AUTOTUNE)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df, training=False)
test_ds = create_dataset(test_df, training=False)

print("Datasets recreated successfully with clipping.")
print("Train batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Test batches:", len(test_ds))

In [ ]:
# STEP 8 - Build Custom CNN baseline model

from tensorflow.keras import layers, models

def build_custom_cnn(input_shape=(224, 224, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.50),
        layers.Dense(1, activation="sigmoid")
    ])

    return model

custom_cnn = build_custom_cnn()

custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

custom_cnn.summary()

In [ ]:
# STEP 9 - Train Custom CNN baseline model

callbacks_custom = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/custom_cnn_best.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7
    )
]

EPOCHS = 30

history_custom = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_custom
)

In [ ]:
# STEP 10 - Plot Custom CNN training curves

history_df = pd.DataFrame(history_custom.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Custom CNN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Custom CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["auc"], label="Training AUC")
plt.plot(history_df["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Custom CNN Training and Validation AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# STEP 11 - Evaluate Custom CNN on test set

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

custom_cnn_best = tf.keras.models.load_model("/content/custom_cnn_best.keras")

y_true = []
y_prob = []

for_images = []

for images, labels in test_ds:
    probs = custom_cnn_best.predict(images, verbose=0).ravel()
    y_prob.extend(probs)
    y_true.extend(labels.numpy())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
specificity = tn / (tn + fp)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)

custom_results = {
    "Model": "Custom CNN",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall/Sensitivity": recall,
    "Specificity": specificity,
    "F1-score": f1,
    "ROC-AUC": roc_auc,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

print("Custom CNN Test Results")
for key, value in custom_results.items():
    print(f"{key}: {value}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Abnormal"]))

In [ ]:
# STEP 12 - Plot confusion matrix for Custom CNN

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Custom CNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Normal", "Abnormal"])
plt.yticks([0, 1], ["Normal", "Abnormal"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

plt.colorbar()
plt.show()

In [ ]:
# STEP 13 - Save Custom CNN result for final comparison

all_results = []

all_results.append(custom_results)

results_df = pd.DataFrame(all_results)
results_df

In [ ]:
# STEP 14 - Build ResNet50 transfer learning model

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras import layers, models

def build_resnet50_model(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)

    # Dataset images are in [0,1]. ResNet50 expects ImageNet-style preprocessing.
    x = layers.Lambda(lambda img: resnet_preprocess(img * 255.0))(inputs)

    base_model = ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )

    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="ResNet50_Transfer_Learning")

    return model, base_model

resnet50_model, resnet50_base = build_resnet50_model()

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

resnet50_model.summary()

In [ ]:
# STEP 15 - Train ResNet50 with frozen base

callbacks_resnet50_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/resnet50_stage1_best.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7
    )
]

history_resnet50_stage1 = resnet50_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_resnet50_stage1
)

In [ ]:
# STEP 16 - Fine-tune selected deeper layers of ResNet50

resnet50_base.trainable = True


for layer in resnet50_base.layers[:-30]:
    layer.trainable = False

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

callbacks_resnet50_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/resnet50_best.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-8
    )
]

history_resnet50_stage2 = resnet50_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_resnet50_stage2
)

In [ ]:
# STEP 17 - Combine and plot ResNet50 training curves

history_resnet50 = {}

for key in history_resnet50_stage1.history.keys():
    history_resnet50[key] = (
        history_resnet50_stage1.history[key] +
        history_resnet50_stage2.history[key]
    )

history_resnet50_df = pd.DataFrame(history_resnet50)

plt.figure(figsize=(8, 5))
plt.plot(history_resnet50_df["loss"], label="Training Loss")
plt.plot(history_resnet50_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_resnet50_df["accuracy"], label="Training Accuracy")
plt.plot(history_resnet50_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet50 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_resnet50_df["auc"], label="Training AUC")
plt.plot(history_resnet50_df["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("ResNet50 Training and Validation AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# STEP 18 - Evaluate ResNet50 on test set WITHOUT loading the saved model

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import numpy as np
import pandas as pd

# Model already exists in memory after training and fine-tuning
resnet50_best = resnet50_model

y_true_resnet = []
y_prob_resnet = []

for images, labels in test_ds:
    probs = resnet50_best.predict(images, verbose=0).ravel()
    y_prob_resnet.extend(probs)
    y_true_resnet.extend(labels.numpy())

y_true_resnet = np.array(y_true_resnet).astype(int)
y_prob_resnet = np.array(y_prob_resnet)
y_pred_resnet = (y_prob_resnet >= 0.5).astype(int)

cm_resnet = confusion_matrix(y_true_resnet, y_pred_resnet)
tn, fp, fn, tp = cm_resnet.ravel()

resnet_accuracy = accuracy_score(y_true_resnet, y_pred_resnet)
resnet_precision = precision_score(y_true_resnet, y_pred_resnet)
resnet_recall = recall_score(y_true_resnet, y_pred_resnet)
resnet_specificity = tn / (tn + fp)
resnet_f1 = f1_score(y_true_resnet, y_pred_resnet)
resnet_roc_auc = roc_auc_score(y_true_resnet, y_prob_resnet)

resnet50_results = {
    "Model": "ResNet50",
    "Accuracy": resnet_accuracy,
    "Precision": resnet_precision,
    "Recall/Sensitivity": resnet_recall,
    "Specificity": resnet_specificity,
    "F1-score": resnet_f1,
    "ROC-AUC": resnet_roc_auc,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

print("ResNet50 Test Results")
for key, value in resnet50_results.items():
    print(f"{key}: {value}")

print("\nClassification Report:")
print(classification_report(y_true_resnet, y_pred_resnet, target_names=["Normal", "Abnormal"]))

In [ ]:
# STEP 19 - Plot confusion matrix for ResNet50

plt.figure(figsize=(6, 5))
plt.imshow(cm_resnet)
plt.title("ResNet50 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Normal", "Abnormal"])
plt.yticks([0, 1], ["Normal", "Abnormal"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_resnet[i, j], ha="center", va="center", fontsize=12)

plt.colorbar()
plt.show()

In [ ]:
# STEP 20 - Build EfficientNetB0 transfer learning model

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

def build_efficientnetb0_model(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)


    x = layers.Lambda(lambda img: img * 255.0)(inputs)

    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )

    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_Transfer_Learning")

    return model, base_model

efficientnet_model, efficientnet_base = build_efficientnetb0_model()

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

efficientnet_model.summary()

In [ ]:
# STEP 21 - Train EfficientNetB0 with frozen base

callbacks_efficientnet_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7
    )
]

history_efficientnet_stage1 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_efficientnet_stage1
)

In [ ]:
# STEP 22 - Fine-tune selected deeper layers of EfficientNetB0

efficientnet_base.trainable = True


for layer in efficientnet_base.layers[:-30]:
    layer.trainable = False

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

callbacks_efficientnet_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-8
    )
]

history_efficientnet_stage2 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_efficientnet_stage2
)

In [ ]:
# STEP 23 - Combine and plot EfficientNetB0 training curves

history_efficientnet = {}

for key in history_efficientnet_stage1.history.keys():
    history_efficientnet[key] = (
        history_efficientnet_stage1.history[key] +
        history_efficientnet_stage2.history[key]
    )

history_efficientnet_df = pd.DataFrame(history_efficientnet)

plt.figure(figsize=(8, 5))
plt.plot(history_efficientnet_df["loss"], label="Training Loss")
plt.plot(history_efficientnet_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNetB0 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_efficientnet_df["accuracy"], label="Training Accuracy")
plt.plot(history_efficientnet_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNetB0 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_efficientnet_df["auc"], label="Training AUC")
plt.plot(history_efficientnet_df["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("EfficientNetB0 Training and Validation AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# STEP 24 - Evaluate EfficientNetB0 on test set

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import numpy as np

efficientnet_best = efficientnet_model

y_true_eff = []
y_prob_eff = []

for images, labels in test_ds:
    probs = efficientnet_best.predict(images, verbose=0).ravel()
    y_prob_eff.extend(probs)
    y_true_eff.extend(labels.numpy())

y_true_eff = np.array(y_true_eff).astype(int)
y_prob_eff = np.array(y_prob_eff)
y_pred_eff = (y_prob_eff >= 0.5).astype(int)

cm_eff = confusion_matrix(y_true_eff, y_pred_eff)
tn, fp, fn, tp = cm_eff.ravel()

eff_accuracy = accuracy_score(y_true_eff, y_pred_eff)
eff_precision = precision_score(y_true_eff, y_pred_eff)
eff_recall = recall_score(y_true_eff, y_pred_eff)
eff_specificity = tn / (tn + fp)
eff_f1 = f1_score(y_true_eff, y_pred_eff)
eff_roc_auc = roc_auc_score(y_true_eff, y_prob_eff)

efficientnet_results = {
    "Model": "EfficientNetB0",
    "Accuracy": eff_accuracy,
    "Precision": eff_precision,
    "Recall/Sensitivity": eff_recall,
    "Specificity": eff_specificity,
    "F1-score": eff_f1,
    "ROC-AUC": eff_roc_auc,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

print("EfficientNetB0 Test Results")
for key, value in efficientnet_results.items():
    print(f"{key}: {value}")

print("\nClassification Report:")
print(classification_report(y_true_eff, y_pred_eff, target_names=["Normal", "Abnormal"]))

In [ ]:
# STEP 25 - Plot confusion matrix for EfficientNetB0

plt.figure(figsize=(6, 5))
plt.imshow(cm_eff)
plt.title("EfficientNetB0 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Normal", "Abnormal"])
plt.yticks([0, 1], ["Normal", "Abnormal"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_eff[i, j], ha="center", va="center", fontsize=12)

plt.colorbar()
plt.show()

In [ ]:
# STEP 26 - Final model comparison table

all_results = [
    custom_results,
    resnet50_results,
    efficientnet_results
]

results_df = pd.DataFrame(all_results)

metric_cols = [
    "Accuracy",
    "Precision",
    "Recall/Sensitivity",
    "Specificity",
    "F1-score",
    "ROC-AUC"
]

results_display = results_df.copy()

for col in metric_cols:
    results_display[col] = results_display[col].round(4)

results_display

In [ ]:
# STEP 27 - Bar chart comparison of model metrics

plot_df = results_df.set_index("Model")[metric_cols]

ax = plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Comparison of Model Performance on the Test Set")
plt.ylabel("Score")
plt.ylim(0.85, 1.01)
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# STEP 28 - ROC curve comparison for all models

from sklearn.metrics import roc_curve, auc

fpr_custom, tpr_custom, _ = roc_curve(y_true, y_prob)
fpr_resnet, tpr_resnet, _ = roc_curve(y_true_resnet, y_prob_resnet)
fpr_eff, tpr_eff, _ = roc_curve(y_true_eff, y_prob_eff)

auc_custom = roc_auc_score(y_true, y_prob)
auc_resnet = roc_auc_score(y_true_resnet, y_prob_resnet)
auc_eff = roc_auc_score(y_true_eff, y_prob_eff)

plt.figure(figsize=(8, 6))
plt.plot(fpr_custom, tpr_custom, label=f"Custom CNN AUC = {auc_custom:.4f}")
plt.plot(fpr_resnet, tpr_resnet, label=f"ResNet50 AUC = {auc_resnet:.4f}")
plt.plot(fpr_eff, tpr_eff, label=f"EfficientNetB0 AUC = {auc_eff:.4f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison of CNN Models")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 29 - Select best model based on overall metrics

best_model_name = results_df.sort_values(
    by=["ROC-AUC", "F1-score", "Accuracy", "Recall/Sensitivity"],
    ascending=False
).iloc[0]["Model"]

print("Best model based on overall performance:", best_model_name)

results_df.sort_values(
    by=["ROC-AUC", "F1-score", "Accuracy", "Recall/Sensitivity"],
    ascending=False
)

In [ ]:
# STEP 30 - Prepare test dataframe for Grad-CAM using the Custom CNN model

gradcam_model = custom_cnn_best

test_results_custom_df = test_df.reset_index(drop=True).copy()
test_results_custom_df["true_label"] = y_true
test_results_custom_df["pred_prob"] = y_prob
test_results_custom_df["pred_label"] = y_pred
test_results_custom_df["correct"] = test_results_custom_df["true_label"] == test_results_custom_df["pred_label"]

print("Correct predictions:", test_results_custom_df["correct"].sum())
print("Incorrect predictions:", (~test_results_custom_df["correct"]).sum())

test_results_custom_df.head()

In [ ]:
# STEP 31 - Grad-CAM helper functions for Sequential Custom CNN

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

gradcam_model = custom_cnn_best


_ = gradcam_model(tf.zeros((1, 224, 224, 3)), training=False)

def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found in the model.")

last_conv_layer_name = find_last_conv_layer(gradcam_model)
print("Last convolutional layer:", last_conv_layer_name)


def load_image_for_gradcam(path, img_size=(224, 224)):
    img = tf.keras.utils.load_img(path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = img_array / 255.0
    return img_array.astype("float32")


def call_layer_safely(layer, x):

    if isinstance(layer, (tf.keras.layers.BatchNormalization, tf.keras.layers.Dropout)):
        return layer(x, training=False)
    return layer(x)


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, target_class):
    img_batch = tf.convert_to_tensor(np.expand_dims(img_array, axis=0), dtype=tf.float32)

    with tf.GradientTape() as tape:
        x = img_batch
        conv_outputs = None

        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.InputLayer):
                continue

            x = call_layer_safely(layer, x)

            if layer.name == last_conv_layer_name:
                conv_outputs = x
                tape.watch(conv_outputs)

        predictions = x

        if target_class == 1:
            loss = predictions[:, 0]
        else:
            loss = 1.0 - predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def display_gradcam(path, true_label, pred_label, pred_prob, model, last_conv_layer_name):
    img_array = load_image_for_gradcam(path)

    heatmap = make_gradcam_heatmap(
        img_array=img_array,
        model=model,
        last_conv_layer_name=last_conv_layer_name,
        target_class=pred_label
    )

    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (img_array.shape[0], img_array.shape[1])
    ).numpy().squeeze()

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img_array)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img_array)
    plt.imshow(heatmap_resized, cmap="jet", alpha=0.4)
    plt.title("Overlay")
    plt.axis("off")

    true_name = "Abnormal" if true_label == 1 else "Normal"
    pred_name = "Abnormal" if pred_label == 1 else "Normal"

    plt.suptitle(
        f"True: {true_name} | Predicted: {pred_name} | Probability: {pred_prob:.4f}",
        fontsize=12
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 32 - Grad-CAM for correctly classified examples

correct_examples = test_results_custom_df[test_results_custom_df["correct"] == True]

correct_normal = correct_examples[correct_examples["true_label"] == 0].sample(2, random_state=SEED)
correct_abnormal = correct_examples[correct_examples["true_label"] == 1].sample(2, random_state=SEED)

selected_correct = pd.concat([correct_normal, correct_abnormal])

for row in selected_correct.itertuples():
    display_gradcam(
        path=row.path,
        true_label=row.true_label,
        pred_label=row.pred_label,
        pred_prob=row.pred_prob,
        model=gradcam_model,
        last_conv_layer_name=last_conv_layer_name
    )

In [ ]:
# STEP 33 - Grad-CAM for incorrectly classified examples

incorrect_examples = test_results_custom_df[test_results_custom_df["correct"] == False]

print("Number of incorrect examples:", len(incorrect_examples))

selected_incorrect = incorrect_examples.sample(
    min(4, len(incorrect_examples)),
    random_state=SEED
)

for row in selected_incorrect.itertuples():
    display_gradcam(
        path=row.path,
        true_label=row.true_label,
        pred_label=row.pred_label,
        pred_prob=row.pred_prob,
        model=gradcam_model,
        last_conv_layer_name=last_conv_layer_name
    )